# Recommendation Message Experiment

## Purpose

사용자의 건강 분석 결과와 추천 챌린지를 기반으로 건강관리 추천 문구를 생성하는 LLM 프롬프트를 실험한다.

MVP에서 추천 문구 생성 기능은 진단, 처방, 치료 지시를 수행하지 않는다.  
역할은 건강관리 참고용 안내와 챌린지 참여 유도 문구 생성으로 제한한다.

## Experiment Goal

1. 추천 문구 생성에 필요한 입력 데이터 구조를 정의한다.
2. LLM 프롬프트 초안을 작성한다.
3. Mock Generator로 LLM 없이 기본 출력 구조를 테스트한다.
4. 금지 표현 Safety Check를 적용한다.
5. 위험도/챌린지 조합별 테스트 케이스를 비교한다.
6. 최종 후보 프롬프트를 정리한다.

In [1]:
import sys
import os

print(sys.executable)
print(os.getcwd())

/Users/admin/PycharmProjects/AH_03_03/.venv/bin/python
/Users/admin/PycharmProjects/AH_03_03/ai_worker/llm/notebooks


In [2]:
import json
from copy import deepcopy
from pprint import pprint
import os
from pathlib import Path

from dotenv import load_dotenv
from openai import OpenAI

def find_env_file(start: Path = Path.cwd()) -> Path | None:
    for path in [start, *start.parents]:
        for candidate in (path / ".env", path / "envs" / ".local.env"):
            if candidate.exists():
                return candidate
    return None


env_path = find_env_file()

if not env_path:
    raise FileNotFoundError(".env 또는 envs/.local.env 파일을 찾지 못했습니다.")

load_dotenv(env_path)

api_key = os.getenv("OPENAI_API_KEY")

if not api_key:
    raise ValueError(f"OPENAI_API_KEY가 없습니다. 확인한 env 파일: {env_path}")

client = OpenAI(api_key=api_key)

print(f"OpenAI client ready: {env_path}")

OpenAI client ready: /Users/admin/PycharmProjects/AH_03_03/.env


In [3]:
sample_input = {
    "user_context": {
        "age_group": "30s",
        "gender": "M"
    },
    "analysis_result": {
        "diabetes_risk_score": 0.72,
        "diabetes_risk_level": "high",
        "hypertension_risk_score": 0.61,
        "hypertension_risk_level": "medium",
        "main_risk_factors": ["BMI", "fasting_glucose", "systolic_bp"]
    },
    "recommended_challenges": [
        {
            "title": "하루 7000보 걷기",
            "reason": "체중 관리와 혈당 관리에 도움이 될 수 있음"
        },
        {
            "title": "단 음료 줄이기",
            "reason": "혈당 관리에 도움이 될 수 있음"
        }
    ]
}

pprint(sample_input)

{'analysis_result': {'diabetes_risk_level': 'high',
                     'diabetes_risk_score': 0.72,
                     'hypertension_risk_level': 'medium',
                     'hypertension_risk_score': 0.61,
                     'main_risk_factors': ['BMI',
                                           'fasting_glucose',
                                           'systolic_bp']},
 'recommended_challenges': [{'reason': '체중 관리와 혈당 관리에 도움이 될 수 있음',
                             'title': '하루 7000보 걷기'},
                            {'reason': '혈당 관리에 도움이 될 수 있음',
                             'title': '단 음료 줄이기'}],
 'user_context': {'age_group': '30s', 'gender': 'M'}}


In [4]:
SYSTEM_PROMPT_V1 = """
너는 만성질환 예방 서비스의 건강관리 추천 문구 생성 AI다.

역할:
- 사용자의 건강정보 분석 결과와 추천 챌린지를 바탕으로 짧고 이해하기 쉬운 안내 문구를 생성한다.
- 사용자가 불안해하지 않도록 부드럽고 현실적인 문장으로 작성한다.
- 진단, 처방, 치료 지시는 하지 않는다.

금지 표현:
- "당뇨입니다"
- "고혈압입니다"
- "치료가 필요합니다"
- "약을 복용하세요"
- "이 방법으로 완치됩니다"
- "반드시 병에 걸립니다"
- "병원에 가지 않아도 됩니다"

허용 표현:
- "위험도가 높게 예측되었습니다"
- "관리가 필요할 수 있습니다"
- "생활습관 개선에 도움이 될 수 있습니다"
- "건강관리 참고용 안내입니다"
- "정확한 진단과 치료는 의료진 상담이 필요합니다"

출력 규칙:
- 반드시 JSON 형식으로만 응답한다.
- markdown 코드블록을 사용하지 않는다.
- 한국어로 작성한다.
- summary_message, challenge_message, caution_message, tone 필드를 반드시 포함한다.
"""

In [5]:
def build_recommendation_prompt(input_data: dict) -> str:
    return f"""
아래 사용자 건강 분석 데이터를 바탕으로 추천 문구를 생성해라.

입력 데이터:
{json.dumps(input_data, ensure_ascii=False, indent=2)}

출력 JSON 형식:
{{
  "summary_message": "분석 결과 요약 문구",
  "challenge_message": "추천 챌린지 안내 문구",
  "caution_message": "진단이 아니라 참고용이라는 주의 문구",
  "tone": "friendly"
}}

작성 조건:
1. summary_message는 1~2문장으로 작성한다.
2. challenge_message는 추천 챌린지를 자연스럽게 연결한다.
3. caution_message에는 반드시 진단이 아니라 건강관리 참고용이라는 의미를 포함한다.
4. 질병을 확정하지 않는다.
5. 사용자를 겁주지 않는다.
6. 치료, 처방, 약물 복용 변경을 지시하지 않는다.
7. 위험도는 "가능성", "관리 필요", "참고용" 수준으로 표현한다.
"""

In [6]:
RISK_LEVEL_KO = {
    "low": "낮음",
    "medium": "보통",
    "high": "높음",
    "very_high": "매우 높음"
}

FACTOR_NAME_KO = {
    "BMI": "BMI",
    "fasting_glucose": "공복혈당",
    "systolic_bp": "수축기 혈압",
    "diastolic_bp": "이완기 혈압",
    "hba1c": "당화혈색소",
    "weight": "체중",
    "waist_cm": "허리둘레",
    "total_cholesterol": "총콜레스테롤",
    "hdl_cholesterol": "HDL 콜레스테롤",
    "ldl_cholesterol": "LDL 콜레스테롤",
    "triglyceride": "중성지방"
}


def translate_factor_name(factor: str) -> str:
    return FACTOR_NAME_KO.get(factor, factor)


def mock_recommendation_message(input_data: dict) -> dict:
    analysis = input_data.get("analysis_result", {})
    challenges = input_data.get("recommended_challenges", [])

    diabetes_level = analysis.get("diabetes_risk_level")
    hypertension_level = analysis.get("hypertension_risk_level")
    main_factors = analysis.get("main_risk_factors", [])

    translated_factors = [translate_factor_name(factor) for factor in main_factors[:3]]

    # 1. summary_message 생성
    if translated_factors:
        factor_text = ", ".join(translated_factors)
        summary_message = (
            f"입력된 건강정보 기준으로 {factor_text} 항목이 "
            "건강 위험도에 영향을 준 것으로 보입니다."
        )
    else:
        summary_message = (
            "입력된 건강정보 기준으로 현재 건강 상태를 꾸준히 관리하는 것이 중요해 보입니다."
        )

    risk_notes = []

    if diabetes_level in ["high", "very_high"]:
        risk_notes.append("혈당 관리")
    elif diabetes_level == "medium":
        risk_notes.append("혈당 상태 점검")

    if hypertension_level in ["high", "very_high"]:
        risk_notes.append("혈압 관리")
    elif hypertension_level == "medium":
        risk_notes.append("혈압 상태 점검")

    if risk_notes:
        summary_message += f" 특히 {'와 '.join(risk_notes)}가 필요할 수 있습니다."

    # 2. challenge_message 생성
    if challenges:
        challenge_titles = [challenge["title"] for challenge in challenges[:2]]
        challenge_message = (
            f"이번 주에는 {'와 '.join(challenge_titles)} 챌린지를 "
            "우선 실천해보는 것을 추천드립니다."
        )
    else:
        challenge_message = (
            "이번 주에는 걷기, 식단 기록, 수분 섭취 같은 기본 건강관리 습관부터 "
            "무리하지 않는 범위에서 시작해보는 것을 추천드립니다."
        )

    # 3. caution_message 생성
    caution_message = (
        "이 결과는 진단이 아니라 입력된 건강정보를 바탕으로 한 건강관리 참고용 안내입니다. "
        "정확한 진단과 치료는 의료진 상담이 필요합니다."
    )

    return {
        "summary_message": summary_message,
        "challenge_message": challenge_message,
        "caution_message": caution_message,
        "tone": "friendly"
    }

In [7]:
FORBIDDEN_EXPRESSIONS = [
    "당뇨입니다",
    "고혈압입니다",
    "치료가 필요합니다",
    "약을 복용하세요",
    "약을 끊으세요",
    "완치됩니다",
    "반드시 병에 걸립니다",
    "병원에 가지 않아도 됩니다",
    "의사와 상담하지 않아도 됩니다",
]


REQUIRED_CAUTION_KEYWORDS = [
    "진단",
    "참고용"
]


def safety_check(output: dict) -> dict:
    combined_text = " ".join(str(value) for value in output.values())

    detected_forbidden = [
        expression for expression in FORBIDDEN_EXPRESSIONS
        if expression in combined_text
    ]

    missing_required_keywords = [
        keyword for keyword in REQUIRED_CAUTION_KEYWORDS
        if keyword not in output.get("caution_message", "")
    ]

    is_safe = len(detected_forbidden) == 0 and len(missing_required_keywords) == 0

    return {
        "is_safe": is_safe,
        "detected_forbidden": detected_forbidden,
        "missing_required_keywords": missing_required_keywords
    }

In [8]:
test_cases = [
    {
        "name": "Case 1. 당뇨 high / 고혈압 medium / 추천 챌린지 있음",
        "input": sample_input
    },
    {
        "name": "Case 2. 당뇨 low / 고혈압 low / 위험요인 없음",
        "input": {
            "user_context": {
                "age_group": "20s",
                "gender": "F"
            },
            "analysis_result": {
                "diabetes_risk_score": 0.18,
                "diabetes_risk_level": "low",
                "hypertension_risk_score": 0.22,
                "hypertension_risk_level": "low",
                "main_risk_factors": []
            },
            "recommended_challenges": [
                {
                    "title": "하루 물 6잔 마시기",
                    "reason": "기본 건강습관 유지"
                }
            ]
        }
    },
    {
        "name": "Case 3. 당뇨 medium / 고혈압 medium / 챌린지 없음",
        "input": {
            "user_context": {
                "age_group": "40s",
                "gender": "M"
            },
            "analysis_result": {
                "diabetes_risk_score": 0.55,
                "diabetes_risk_level": "medium",
                "hypertension_risk_score": 0.48,
                "hypertension_risk_level": "medium",
                "main_risk_factors": ["BMI"]
            },
            "recommended_challenges": []
        }
    },
    {
        "name": "Case 4. 당뇨 very_high / 고혈압 high / 위험요인 다수",
        "input": {
            "user_context": {
                "age_group": "50s",
                "gender": "M"
            },
            "analysis_result": {
                "diabetes_risk_score": 0.86,
                "diabetes_risk_level": "very_high",
                "hypertension_risk_score": 0.78,
                "hypertension_risk_level": "high",
                "main_risk_factors": [
                    "fasting_glucose",
                    "hba1c",
                    "BMI",
                    "systolic_bp"
                ]
            },
            "recommended_challenges": [
                {
                    "title": "단 음료 줄이기",
                    "reason": "혈당 관리에 도움이 될 수 있음"
                },
                {
                    "title": "저염식 실천",
                    "reason": "혈압 관리에 도움이 될 수 있음"
                },
                {
                    "title": "하루 7000보 걷기",
                    "reason": "체중 관리에 도움이 될 수 있음"
                }
            ]
        }
    },
    {
        "name": "Case 5. 분석 결과 일부 누락",
        "input": {
            "user_context": {
                "age_group": None,
                "gender": None
            },
            "analysis_result": {
                "diabetes_risk_score": None,
                "diabetes_risk_level": None,
                "hypertension_risk_score": 0.34,
                "hypertension_risk_level": "medium",
                "main_risk_factors": ["systolic_bp"]
            },
            "recommended_challenges": [
                {
                    "title": "혈압 기록하기",
                    "reason": "혈압 변화를 확인하는 데 도움이 될 수 있음"
                }
            ]
        }
    }
]

In [9]:
experiment_results = []

for case in test_cases:
    output = mock_recommendation_message(case["input"])
    safety = safety_check(output)

    experiment_results.append({
        "case_name": case["name"],
        "input": case["input"],
        "output": output,
        "safety": safety
    })

    print("=" * 100)
    print(case["name"])
    print("\n[Output]")
    print(json.dumps(output, ensure_ascii=False, indent=2))
    print("\n[Safety Check]")
    print(json.dumps(safety, ensure_ascii=False, indent=2))

Case 1. 당뇨 high / 고혈압 medium / 추천 챌린지 있음

[Output]
{
  "summary_message": "입력된 건강정보 기준으로 BMI, 공복혈당, 수축기 혈압 항목이 건강 위험도에 영향을 준 것으로 보입니다. 특히 혈당 관리와 혈압 상태 점검가 필요할 수 있습니다.",
  "challenge_message": "이번 주에는 하루 7000보 걷기와 단 음료 줄이기 챌린지를 우선 실천해보는 것을 추천드립니다.",
  "caution_message": "이 결과는 진단이 아니라 입력된 건강정보를 바탕으로 한 건강관리 참고용 안내입니다. 정확한 진단과 치료는 의료진 상담이 필요합니다.",
  "tone": "friendly"
}

[Safety Check]
{
  "is_safe": true,
  "detected_forbidden": [],
  "missing_required_keywords": []
}
Case 2. 당뇨 low / 고혈압 low / 위험요인 없음

[Output]
{
  "summary_message": "입력된 건강정보 기준으로 현재 건강 상태를 꾸준히 관리하는 것이 중요해 보입니다.",
  "challenge_message": "이번 주에는 하루 물 6잔 마시기 챌린지를 우선 실천해보는 것을 추천드립니다.",
  "caution_message": "이 결과는 진단이 아니라 입력된 건강정보를 바탕으로 한 건강관리 참고용 안내입니다. 정확한 진단과 치료는 의료진 상담이 필요합니다.",
  "tone": "friendly"
}

[Safety Check]
{
  "is_safe": true,
  "detected_forbidden": [],
  "missing_required_keywords": []
}
Case 3. 당뇨 medium / 고혈압 medium / 챌린지 없음

[Output]
{
  "summary_message": "입력된 건강정보 기준으로 BMI 항목이 건강 위험도에 영향을 준 것으로 보입니다. 

In [11]:
try:
    import pandas as pd

    summary_rows = []

    for result in experiment_results:
        output = result["output"]
        safety = result["safety"]

        summary_rows.append({
            "case_name": result["case_name"],
            "summary_message": output["summary_message"],
            "challenge_message": output["challenge_message"],
            "caution_message": output["caution_message"],
            "is_safe": safety["is_safe"],
            "detected_forbidden": ", ".join(safety["detected_forbidden"]),
            "missing_required_keywords": ", ".join(safety["missing_required_keywords"])
        })

    result_df = pd.DataFrame(summary_rows)
    display(result_df)

except ImportError:
    print("pandas가 설치되어 있지 않습니다.")

,case_name,summary_message,challenge_message,caution_message,is_safe,detected_forbidden,missing_required_keywords
0,Case 1. 당뇨 high / 고혈압 medium / 추천 챌린지 있음,"입력된 건강정보 기준으로 BMI, 공복혈당, 수축기 혈압 항목이 건강 위험도에 영향...",이번 주에는 하루 7000보 걷기와 단 음료 줄이기 챌린지를 우선 실천해보는 것을 ...,이 결과는 진단이 아니라 입력된 건강정보를 바탕으로 한 건강관리 참고용 안내입니다....,True,,
1,Case 2. 당뇨 low / 고혈압 low / 위험요인 없음,입력된 건강정보 기준으로 현재 건강 상태를 꾸준히 관리하는 것이 중요해 보입니다.,이번 주에는 하루 물 6잔 마시기 챌린지를 우선 실천해보는 것을 추천드립니다.,이 결과는 진단이 아니라 입력된 건강정보를 바탕으로 한 건강관리 참고용 안내입니다....,True,,
2,Case 3. 당뇨 medium / 고혈압 medium / 챌린지 없음,입력된 건강정보 기준으로 BMI 항목이 건강 위험도에 영향을 준 것으로 보입니다. ...,"이번 주에는 걷기, 식단 기록, 수분 섭취 같은 기본 건강관리 습관부터 무리하지 않...",이 결과는 진단이 아니라 입력된 건강정보를 바탕으로 한 건강관리 참고용 안내입니다....,True,,
3,Case 4. 당뇨 very_high / 고혈압 high / 위험요인 다수,"입력된 건강정보 기준으로 공복혈당, 당화혈색소, BMI 항목이 건강 위험도에 영향을...",이번 주에는 단 음료 줄이기와 저염식 실천 챌린지를 우선 실천해보는 것을 추천드립니다.,이 결과는 진단이 아니라 입력된 건강정보를 바탕으로 한 건강관리 참고용 안내입니다....,True,,
4,Case 5. 분석 결과 일부 누락,입력된 건강정보 기준으로 수축기 혈압 항목이 건강 위험도에 영향을 준 것으로 보입니...,이번 주에는 혈압 기록하기 챌린지를 우선 실천해보는 것을 추천드립니다.,이 결과는 진단이 아니라 입력된 건강정보를 바탕으로 한 건강관리 참고용 안내입니다....,True,,


In [12]:
unsafe_output = {
    "summary_message": "당뇨입니다. 치료가 필요합니다.",
    "challenge_message": "약을 복용하세요.",
    "caution_message": "참고용 안내입니다.",
    "tone": "friendly"
}

safety_check(unsafe_output)

{'is_safe': False,
 'detected_forbidden': ['당뇨입니다', '치료가 필요합니다', '약을 복용하세요'],
 'missing_required_keywords': ['진단']}

In [13]:
{
    "is_safe": False,
    "detected_forbidden": ["당뇨입니다", "치료가 필요합니다", "약을 복용하세요"],
    "missing_required_keywords": ["진단"]
}

{'is_safe': False,
 'detected_forbidden': ['당뇨입니다', '치료가 필요합니다', '약을 복용하세요'],
 'missing_required_keywords': ['진단']}

In [14]:
# 실제 LLM API 호출 테스트 - 샘플 1개만

def call_llm_recommendation_message(input_data: dict, model: str = "gpt-4o-mini") -> dict:
    user_prompt = build_recommendation_prompt(input_data)

    response = client.responses.create(
        model=model,
        input=[
            {
                "role": "system",
                "content": SYSTEM_PROMPT_V1,
            },
            {
                "role": "user",
                "content": user_prompt,
            },
        ],
    )

    response_text = response.output_text

    try:
        parsed = json.loads(response_text)
    except json.JSONDecodeError as exc:
        print("LLM raw response:")
        print(response_text)
        raise ValueError("LLM 응답이 JSON 형식이 아닙니다.") from exc

    return parsed

In [15]:
llm_result = call_llm_recommendation_message(sample_input)

print(json.dumps(llm_result, ensure_ascii=False, indent=2))

{
  "summary_message": "분석 결과, 관리가 필요할 수 있는 건강요소가 발견되었습니다. 이를 통해 보다 건강한 생활을 도모해보세요.",
  "challenge_message": "하루 7000보 걷기와 단 음료 줄이기를 도전해보세요. 이러한 습관이 체중 및 혈당 관리에 도움이 될 수 있습니다.",
  "caution_message": "이 정보는 진단이 아닌 건강관리 참고용 안내입니다. 보다 정확한 진단과 치료는 의료진 상담이 필요합니다.",
  "tone": "friendly"
}


# Final Prompt Candidate v1

## System Prompt

너는 만성질환 예방 서비스의 건강관리 추천 문구 생성 AI다.

역할:
- 사용자의 건강정보 분석 결과와 추천 챌린지를 바탕으로 짧고 이해하기 쉬운 안내 문구를 생성한다.
- 사용자가 불안해하지 않도록 부드럽고 현실적인 문장으로 작성한다.
- 진단, 처방, 치료 지시는 하지 않는다.

금지 표현:
- "당뇨입니다"
- "고혈압입니다"
- "치료가 필요합니다"
- "약을 복용하세요"
- "약을 끊으세요"
- "이 방법으로 완치됩니다"
- "반드시 병에 걸립니다"
- "병원에 가지 않아도 됩니다"
- "의사와 상담하지 않아도 됩니다"

허용 표현:
- "위험도가 높게 예측되었습니다"
- "관리가 필요할 수 있습니다"
- "생활습관 개선에 도움이 될 수 있습니다"
- "건강관리 참고용 안내입니다"
- "정확한 진단과 치료는 의료진 상담이 필요합니다"

출력 규칙:
- 반드시 JSON 형식으로만 응답한다.
- markdown 코드블록을 사용하지 않는다.
- 한국어로 작성한다.
- summary_message, challenge_message, caution_message, tone 필드를 반드시 포함한다.

## User Prompt Template

아래 사용자 건강 분석 데이터를 바탕으로 추천 문구를 생성한다.

입력 데이터:
- user_context
- analysis_result
- recommended_challenges

출력 JSON:
{
  "summary_message": "분석 결과 요약 문구",
  "challenge_message": "추천 챌린지 안내 문구",
  "caution_message": "진단이 아니라 참고용이라는 주의 문구",
  "tone": "friendly"
}

작성 조건:
1. summary_message는 1~2문장으로 작성한다.
2. challenge_message는 추천 챌린지를 자연스럽게 연결한다.
3. caution_message에는 반드시 진단이 아니라 건강관리 참고용이라는 의미를 포함한다.
4. 질병을 확정하지 않는다.
5. 사용자를 겁주지 않는다.
6. 치료, 처방, 약물 복용 변경을 지시하지 않는다.
7. 위험도는 "가능성", "관리 필요", "참고용" 수준으로 표현한다.

# Next Step

## 실험 결과

- 추천 문구 생성 기능은 입력 데이터 구조가 고정되어 있어 LLM 적용 난이도가 낮다.
- Mock Generator만으로도 기본적인 추천 문구 생성 흐름을 재현할 수 있다.
- Safety Check를 통해 진단, 처방, 치료 지시성 표현을 1차 필터링할 수 있다.

## 다음 작업

1. 실제 LLM API 연결 전까지 Mock Generator 기반으로 화면/백엔드 연동 구조를 검증한다.
2. 추천 문구 출력 JSON schema를 확정한다.
3. 추천 챌린지 데이터 구조와 연결한다.
4. 실제 LLM API 연결 시 Prompt v1을 사용해 응답 안정성을 테스트한다.
5. 실험이 안정화되면 `ai_worker/llm/recommendation/` 하위의 `.py` 모듈로 분리한다.